# Bilinear Demosaicing from Scratch

**Bloom Series — Article 1 Companion Notebook**

*Every Photo You've Taken Is a Lie — Here's the Math That Reconstructs It*

---

This notebook implements bilinear demosaicing from scratch — no OpenCV, no library calls for the core algorithm. We start with a full-color image, simulate what a raw Bayer sensor would capture, then reconstruct the color image pixel by pixel.

**What you'll build:**
1. A Bayer mosaic simulator (strip an RGB image down to raw sensor data)
2. A bilinear demosaicing algorithm (reconstruct the full-color image)
3. Quality metrics and error analysis (PSNR, per-channel error, edge artifacts)

**Prerequisites:** Python 3, NumPy, Matplotlib, Pillow

**Reference:** Szeliski, *Computer Vision: Algorithms and Applications* (2nd ed.), Section 2.3 — [free PDF](https://szeliski.org/Book/)

## 1. Setup

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

## 2. Load and prepare the image

We start with a full-color RGB image — in this case, a photo from my garden. Your phone's ISP already demosaiced this image when you took it. Our job is to **undo** that (simulate the raw sensor), then **redo** it (demosaic from scratch) and see how close we get.

We resize to 1024×1024 so the pixel-by-pixel loop runs quickly (~30 seconds). The algorithm works at any resolution.

In [ ]:
# Replace with your own image path
IMAGE_PATH = "garden.jpg"

pil_img = Image.open(IMAGE_PATH).convert("RGB")
print(f"Original size: {pil_img.size[0]}×{pil_img.size[1]}")

# Resize for fast processing
pil_img = pil_img.resize((1024, 1024), Image.LANCZOS)
img = np.array(pil_img)
H, W, _ = img.shape
print(f"Working size: {W}×{H}")

plt.imshow(img)
plt.title("Original image (ground truth)")
plt.axis('off')
plt.show()

## 3. The Bayer Color Filter Array

Digital camera sensors are **monochrome** — each pixel measures intensity, not color. To capture color, a mosaic of tiny colored filters (the Bayer CFA) sits over the sensor. Each pixel only sees one color:

```
┌───┬───┬───┬───┐
│ R │ G │ R │ G │  ← Even rows
├───┼───┼───┼───┤
│ G │ B │ G │ B │  ← Odd rows
├───┼───┼───┼───┤
│ R │ G │ R │ G │
├───┼───┼───┼───┤
│ G │ B │ G │ B │
└───┴───┴───┴───┘
```

There are **twice as many green** filters as red or blue — because human eyes are most sensitive to green light (Szeliski §2.3). This gives the final image better perceived sharpness.

The raw sensor output is a **single-channel** array. Two-thirds of the color information is missing and must be reconstructed. That reconstruction is called **demosaicing**.

## 4. Simulate the Bayer mosaic

We take our full-color image and strip it down to what the sensor would have actually captured — one color channel per pixel, following the RGGB pattern.

In [ ]:
def simulate_bayer(rgb_image):
    """
    Simulate raw Bayer sensor output from a full-color image.
    
    Takes an RGB image and keeps only one color channel per pixel,
    following the RGGB Bayer pattern:
        (even row, even col) → Red
        (even row, odd col)  → Green
        (odd row, even col)  → Green
        (odd row, odd col)   → Blue
    
    Returns:
        bayer: single-channel array (H, W) — the raw sensor data
        bayer_color: RGB visualization (H, W, 3) — color-coded for display
    """
    H, W, _ = rgb_image.shape
    bayer = np.zeros((H, W), dtype=np.float64)
    
    # Each pixel keeps only the channel its Bayer filter allows through
    bayer[0::2, 0::2] = rgb_image[0::2, 0::2, 0]   # Red
    bayer[0::2, 1::2] = rgb_image[0::2, 1::2, 1]   # Green (on red rows)
    bayer[1::2, 0::2] = rgb_image[1::2, 0::2, 1]   # Green (on blue rows)
    bayer[1::2, 1::2] = rgb_image[1::2, 1::2, 2]   # Blue
    
    # Color-coded visualization: show each pixel in its filter color
    bayer_color = np.zeros((H, W, 3), dtype=np.uint8)
    bayer_color[0::2, 0::2, 0] = bayer[0::2, 0::2]   # Red channel only
    bayer_color[0::2, 1::2, 1] = bayer[0::2, 1::2]   # Green channel only
    bayer_color[1::2, 0::2, 1] = bayer[1::2, 0::2]   # Green channel only
    bayer_color[1::2, 1::2, 2] = bayer[1::2, 1::2]   # Blue channel only
    
    return bayer, bayer_color

bayer, bayer_color = simulate_bayer(img)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(bayer_color)
axes[0].set_title("Raw Bayer mosaic\n(what the sensor actually captures)")
axes[0].axis('off')
axes[1].imshow(img)
axes[1].set_title("Full-color RGB\n(what your phone shows you)")
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f"Bayer array shape: {bayer.shape} — single channel, one color per pixel")
print(f"Original image shape: {img.shape} — three channels, full color")

**Why does the mosaic look so green?** Because 2 out of every 4 pixels are green filters. Half the sensor is measuring green light, so when we visualize the raw data color-coded, green dominates. This is by design — it matches human visual sensitivity.

## 5. Bilinear demosaicing — from scratch

The simplest demosaicing algorithm: for each pixel, estimate the missing color channels by **averaging the nearest neighbors** that have those colors.

Each position in the 2×2 Bayer tile has a different neighborhood pattern:

**Red pixel** `(even row, even col)` — knows R, needs G and B:
- Green: average of 4 cardinal neighbors (↑↓←→)
- Blue: average of 4 diagonal neighbors (↗↘↙↖)

**Green pixel on red row** `(even row, odd col)` — knows G, needs R and B:
- Red: average of 2 horizontal neighbors (←→)
- Blue: average of 2 vertical neighbors (↑↓)

**Green pixel on blue row** `(odd row, even col)` — knows G, needs R and B:
- Red: average of 2 vertical neighbors (↑↓)
- Blue: average of 2 horizontal neighbors (←→)

**Blue pixel** `(odd row, odd col)` — knows B, needs R and G:
- Red: average of 4 diagonal neighbors (↗↘↙↖)
- Green: average of 4 cardinal neighbors (↑↓←→)

In [ ]:
def bilinear_demosaic(bayer):
    """
    Reconstruct a full-color RGB image from a raw Bayer mosaic
    using bilinear interpolation.
    
    For each pixel, the known color channel is kept and the two
    missing channels are estimated by averaging the nearest
    neighbors that measured those colors.
    
    Args:
        bayer: single-channel array (H, W) with RGGB Bayer pattern
    
    Returns:
        RGB image (H, W, 3) as uint8
    """
    H, W = bayer.shape
    
    # Pad with 1-pixel reflected border so edge pixels have valid neighbors
    padded = np.pad(bayer, 1, mode='reflect')
    out = np.zeros((H, W, 3), dtype=np.float64)
    
    for y in range(H):
        for x in range(W):
            py, px = y + 1, x + 1  # shift into padded coordinates
            
            if y % 2 == 0 and x % 2 == 0:
                # RED pixel — we know R, need G and B
                out[y, x, 0] = padded[py, px]
                out[y, x, 1] = (padded[py-1, px] + padded[py+1, px] +
                                padded[py, px-1] + padded[py, px+1]) / 4
                out[y, x, 2] = (padded[py-1, px-1] + padded[py-1, px+1] +
                                padded[py+1, px-1] + padded[py+1, px+1]) / 4
                
            elif y % 2 == 0 and x % 2 == 1:
                # GREEN pixel (red row) — we know G, need R and B
                out[y, x, 0] = (padded[py, px-1] + padded[py, px+1]) / 2
                out[y, x, 1] = padded[py, px]
                out[y, x, 2] = (padded[py-1, px] + padded[py+1, px]) / 2
                
            elif y % 2 == 1 and x % 2 == 0:
                # GREEN pixel (blue row) — we know G, need R and B
                out[y, x, 0] = (padded[py-1, px] + padded[py+1, px]) / 2
                out[y, x, 1] = padded[py, px]
                out[y, x, 2] = (padded[py, px-1] + padded[py, px+1]) / 2
                
            else:
                # BLUE pixel — we know B, need R and G
                out[y, x, 0] = (padded[py-1, px-1] + padded[py-1, px+1] +
                                padded[py+1, px-1] + padded[py+1, px+1]) / 4
                out[y, x, 1] = (padded[py-1, px] + padded[py+1, px] +
                                padded[py, px-1] + padded[py, px+1]) / 4
                out[y, x, 2] = padded[py, px]
    
    return np.clip(out, 0, 255).astype(np.uint8)

print("Running bilinear demosaicing...")
reconstructed = bilinear_demosaic(bayer)
print(f"Done! Output shape: {reconstructed.shape}")

## 6. Results — how close did we get?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img)
axes[0].set_title("Ground truth")
axes[0].axis('off')

axes[1].imshow(reconstructed)
axes[1].set_title("Bilinear demosaicing")
axes[1].axis('off')

# Error map — amplified 5× to make differences visible
error = np.abs(img.astype(float) - reconstructed.astype(float))
error_vis = np.clip(error * 5, 0, 255).astype(np.uint8)
axes[2].imshow(error_vis)
axes[2].set_title("Error map (×5)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 7. Quality metrics — PSNR

**Peak Signal-to-Noise Ratio (PSNR)** measures reconstruction quality in decibels. Higher is better. For demosaicing, typical baselines:
- Bilinear: ~28–33 dB
- Advanced classical (Malvar et al.): ~33–37 dB  
- Neural methods: ~38+ dB

In [ ]:
def compute_psnr(original, reconstructed):
    """Compute PSNR between two images in dB."""
    mse = np.mean((original.astype(float) - reconstructed.astype(float)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255**2 / mse)

psnr_overall = compute_psnr(img, reconstructed)
print(f"Overall PSNR: {psnr_overall:.2f} dB")

# Per-channel PSNR — green should be highest (2× samples in Bayer pattern)
for i, name in enumerate(["Red", "Green", "Blue"]):
    ch_psnr = compute_psnr(img[:,:,i], reconstructed[:,:,i])
    print(f"  {name}: {ch_psnr:.2f} dB")

**Note:** Green PSNR should be noticeably higher than Red or Blue. That's the Bayer pattern at work — with twice as many green samples, there's more information to interpolate from, so the green channel reconstruction is more accurate.

## 8. Where bilinear fails — edge artifacts

Bilinear demosaicing averages neighbors **without knowing whether they belong to the same object.** At edges — a leaf against sky, a stem against soil — it averages pixels from both sides of the boundary. The result: color bleeds across edges.

This is called **color fringing** or the **zipper effect**. Let's zoom into an edge region to see it clearly.

In [ ]:
# Pick a crop region with a strong edge
# Adjust these coordinates for your image to find a good leaf/sky or plant/soil boundary
y1, y2 = H // 3 - 40, H // 3 + 40
x1, x2 = W // 4, W // 4 + 120

# Fallback: if the auto-pick doesn't look good, try these:
# y1, y2 = 300, 380   # adjust to where you see a strong edge
# x1, x2 = 200, 320

gt_crop = img[y1:y2, x1:x2]
rc_crop = reconstructed[y1:y2, x1:x2]
er_crop = np.clip(np.abs(gt_crop.astype(float) - rc_crop.astype(float)) * 8, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(gt_crop, interpolation='nearest')
axes[0].set_title("Ground truth (zoomed)")
axes[0].axis('off')

axes[1].imshow(rc_crop, interpolation='nearest')
axes[1].set_title("Demosaiced (zoomed)")
axes[1].axis('off')

axes[2].imshow(er_crop, interpolation='nearest')
axes[2].set_title("Error (×8, zoomed)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("Look at the edges — color bleeding across object boundaries.")
print("This is the fundamental limitation that motivated neural demosaicing.")

## 9. Per-channel error visualization

Separating the error by color channel reveals the pattern: green has the least error (most samples), red and blue have the most (fewest samples, longest interpolation distances).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
channel_names = ["Red", "Green", "Blue"]
cmaps = ["Reds", "Greens", "Blues"]

for i in range(3):
    ch_error = np.abs(img[:,:,i].astype(float) - reconstructed[:,:,i].astype(float))
    axes[i].imshow(ch_error, cmap=cmaps[i], vmin=0, vmax=50)
    axes[i].set_title(f"{channel_names[i]} channel error")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 10. What this notebook shows

**Bilinear demosaicing works** — you can reconstruct a convincing full-color image from a single-channel Bayer mosaic using nothing but neighbor averaging.

**But it fails at edges** — because it has no concept of object boundaries. It averages across edges just as happily as within smooth regions, causing color fringing and the zipper effect.

**This is exactly the gap that neural networks fill.** A CNN trained on millions of images learns edge-aware interpolation rules that no hand-designed algorithm could match. The classical-to-neural progression — and the on-device deployment tradeoffs it creates — is the thread running through the entire Bloom series.

---

**Next:** Article 2 — Image Filtering: From Gaussian Blurs to Neural Denoisers

**Full article:** [Every Photo You've Taken Is a Lie — Here's the Math That Reconstructs It](https://medium.com/@rachana.gupta_7569)

**Series intro:** [Bloom: I Couldn't Read My Own Yard. So I'm Building an AI That Can.](https://medium.com/@rachana.gupta_7569/bloom-i-couldnt-read-my-own-yard-so-i-m-building-an-ai-that-can-c97f47723f46)

## References

1. Bayer, B.E. (1976). *Color imaging array.* U.S. Patent 3,971,065.
2. Szeliski, R. (2022). *Computer Vision: Algorithms and Applications* (2nd ed.), Section 2.3. [Free PDF](https://szeliski.org/Book/)
3. Malvar, H.S., He, L., & Cutler, R. (2004). *High-quality linear interpolation for demosaicing of Bayer-patterned color images.* IEEE ICASSP.
4. Chen, C. et al. (2018). *Learning to See in the Dark.* CVPR. [arXiv](https://arxiv.org/abs/1805.01934)

---

*Bloom is a personal side project. All views are my own and unrelated to any employer.*